In [0]:
# the "current state" of our data
target_df = spark.table("sales_line_items_clean")
target_df.createOrReplaceTempView("target_view")
print("Target table rows:", target_df.count())
target_df.show(5)

Target table rows: 7833
+------------+-----------+--------------------+-------------------+--------------------+--------------------+----------+--------+-------------+
|order_number|customer_id|       customer_name|         order_date|          product_id|        product_name|unit_price|quantity|total_revenue|
+------------+-----------+--------------------+-------------------+--------------------+--------------------+----------+--------+-------------+
|   317568020|    7159905|TURNER ALSTON,  D...|2019-08-01 05:36:14|AVpe7vER1cnluZ0-aJu7|Mogitech Keys-To-...|        60|       1|           60|
|   317568020|    7159905|TURNER ALSTON,  D...|2019-08-01 05:36:14|AVpfMVD-ilAPnD_xW6bu|Rony - BC-TRX Bat...|        27|       2|           54|
|   317568038|   22100013|arq electronics m...|2019-08-01 14:35:14|AVpfCW42ilAPnD_xTj0y|CD-C600 5-Disc CD...|       287|       2|          574|
|   317568038|   22100013|arq electronics m...|2019-08-01 14:35:14|AVpfIODe1cnluZ0-eg35|Cyber-shot DSC-WX...|   

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp

# Grab a few real existing rows to simulate "updates" and "cancellations"
sample_rows = target_df.limit(3).collect()

changes = []

# 1. Simulate an UPDATE: same order_number + product_id, but quantity/revenue changed
r = sample_rows[0]
changes.append(Row(
    order_number=r["order_number"], customer_id=r["customer_id"], customer_name=r["customer_name"],
    order_date=r["order_date"], product_id=r["product_id"], product_name=r["product_name"],
    unit_price=r["unit_price"], quantity=r["quantity"] + 5,  # quantity changed
    total_revenue=float(r["unit_price"] * (r["quantity"] + 5)),
    change_type="UPDATE"
))

# 2. Simulate a CANCELLATION (we'll mark it, then delete in the merge)
r2 = sample_rows[1]
changes.append(Row(
    order_number=r2["order_number"], customer_id=r2["customer_id"], customer_name=r2["customer_name"],
    order_date=r2["order_date"], product_id=r2["product_id"], product_name=r2["product_name"],
    unit_price=r2["unit_price"], quantity=r2["quantity"],
    total_revenue=float(r2["total_revenue"]),
    change_type="DELETE"
))

# 3. Simulate a brand NEW order (insert)
changes.append(Row(
    order_number=999999999, customer_id="99999999", customer_name="NEW CUSTOMER TEST",
    order_date=r["order_date"], product_id="NEWPROD123", product_name="Brand New Test Product",
    unit_price=100, quantity=2, total_revenue=200.0,
    change_type="INSERT"
))

changes_df = spark.createDataFrame(changes)
changes_df = changes_df.withColumn("cdc_timestamp", current_timestamp())
changes_df.show(truncate=False)

+------------+-----------+----------------------+-------------------+--------------------+-----------------------------------------------------------------------------+----------+--------+-------------+-----------+--------------------------+
|order_number|customer_id|customer_name         |order_date         |product_id          |product_name                                                                 |unit_price|quantity|total_revenue|change_type|cdc_timestamp             |
+------------+-----------+----------------------+-------------------+--------------------+-----------------------------------------------------------------------------+----------+--------+-------------+-----------+--------------------------+
|317568020   |7159905    |TURNER ALSTON,  DENISE|2019-08-01 05:36:14|AVpe7vER1cnluZ0-aJu7|Mogitech Keys-To-Go Ultra-Portable Bluetooth Keyboard for Android and Windows|60        |6       |360.0        |UPDATE     |2026-07-14 09:17:15.611475|
|317568020   |7159905    |TURNER

In [0]:
from delta.tables import DeltaTable

# Load the target as a DeltaTable object (needed for merge operations)
target_table = DeltaTable.forName(spark, "sales_line_items_clean")

# Register the changes as a temp view for the merge
changes_df.createOrReplaceTempView("changes_view")

# The MERGE: match on order_number + product_id (our natural key)
target_table.alias("target").merge(
    changes_df.alias("source"),
    "target.order_number = source.order_number AND target.product_id = source.product_id"
).whenMatchedDelete(
    condition="source.change_type = 'DELETE'"
).whenMatchedUpdate(
    condition="source.change_type = 'UPDATE'",
    set={
        "quantity": "source.quantity",
        "total_revenue": "source.total_revenue"
    }
).whenNotMatchedInsert(
    condition="source.change_type = 'INSERT'",
    values={
        "order_number": "source.order_number",
        "customer_id": "source.customer_id",
        "customer_name": "source.customer_name",
        "order_date": "source.order_date",
        "product_id": "source.product_id",
        "product_name": "source.product_name",
        "unit_price": "source.unit_price",
        "quantity": "source.quantity",
        "total_revenue": "source.total_revenue"
    }
).execute()

print("Merge complete")

Merge complete


In [0]:
result = spark.table("sales_line_items_clean")
print("New total row count:", result.count())  # should be original count (unchanged net, since 1 delete + 1 insert)

# Check the updated row
result.filter("order_number = 317568020 AND product_id = 'AVpe7vER1cnluZ0-aJu7'").show()

# Check the new inserted row
result.filter("order_number = 999999999").show()

# Confirm the deleted row is gone
result.filter("order_number = 317568020 AND product_id = 'AVpfMVD-ilAPnD_xW6bu'").show()  # should be empty

New total row count: 7833
+------------+-----------+--------------------+-------------------+--------------------+--------------------+----------+--------+-------------+
|order_number|customer_id|       customer_name|         order_date|          product_id|        product_name|unit_price|quantity|total_revenue|
+------------+-----------+--------------------+-------------------+--------------------+--------------------+----------+--------+-------------+
|   317568020|    7159905|TURNER ALSTON,  D...|2019-08-01 05:36:14|AVpe7vER1cnluZ0-aJu7|Mogitech Keys-To-...|        60|       6|          360|
+------------+-----------+--------------------+-------------------+--------------------+--------------------+----------+--------+-------------+

+------------+-----------+-----------------+-------------------+----------+--------------------+----------+--------+-------------+
|order_number|customer_id|    customer_name|         order_date|product_id|        product_name|unit_price|quantity|total_

In [0]:
%sql
DESCRIBE HISTORY sales_line_items_clean

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-07-14T09:18:52.000Z,77190640808905,shivamkumarkaimur@gmail.com,MERGE,"Map(predicate -> [""((order_number#11523L = order_number#11500L) AND (product_id#11527 = product_id#11504))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(change_type#11509 = DELETE)"",""actionType"":""delete""},{""predicate"":""(change_type#11509 = UPDATE)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""predicate"":""(change_type#11509 = INSERT)"",""actionType"":""insert""}])",null,List(2133738795050211),d5f73f10-01ec-44cc-88b3-e8a089971c34,0714-091608-ckh5rrg-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 2, numTargetBytesAdded -> 6086, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 9773, materializeSourceTimeMs -> 442, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 1, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 4183, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4912)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-13T16:29:29.000Z,77190640808905,shivamkumarkaimur@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4028701048855371),00a8b9d9-0dc2-4f57-86bb-3623de906591,0713-162246-q7fg8dxo-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 7833, numOutputBytes -> 140756)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT * FROM sales_line_items_clean VERSION AS OF 0
WHERE order_number = 999999999

order_number,customer_id,customer_name,order_date,product_id,product_name,unit_price,quantity,total_revenue


In [0]:
%sql
SELECT * FROM sales_line_items_clean
WHERE order_number = 999999999

order_number,customer_id,customer_name,order_date,product_id,product_name,unit_price,quantity,total_revenue
999999999,99999999,NEW CUSTOMER TEST,2019-08-01T05:36:14.000Z,NEWPROD123,Brand New Test Product,100,2,200
